In [1]:
print("hola")

hola


In [2]:
# ============================================================
# 1. IMPORTS
# ============================================================

import os
import re
from datetime import date

from dotenv import load_dotenv

from exa_py import Exa

from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.tools import tool

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from langgraph.checkpoint.memory import InMemorySaver

from langchain.agents.middleware import (
    ToolCallLimitMiddleware,
    ModelCallLimitMiddleware,
)


# ============================================================
# 2. VARIABLES DE ENTORNO
# ============================================================

load_dotenv()

openai_api_key = os.getenv("OPENAI_API_KEY")
exa_api_key = os.getenv("EXA_API_KEY")


if not openai_api_key:
    raise ValueError(
        "No se encontró OPENAI_API_KEY en las variables de entorno."
    )

if not exa_api_key:
    raise ValueError(
        "No se encontró EXA_API_KEY en las variables de entorno."
    )

In [4]:
# ============================================================
# 3. CLIENTE EXA Y MODELOS
# ============================================================

exa = Exa(api_key=exa_api_key)


# Modelo principal del agente
llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0,
    api_key=openai_api_key,
    max_retries=2,
)


# Modelo encargado de analizar cada noticia individual
llm_2 = ChatOpenAI(
    model="gpt-5-nano",
    temperature=0,
    api_key=openai_api_key,
    max_retries=2,
)

In [5]:
# ============================================================
# 4. SYSTEM PROMPT - ANALIZADOR INDIVIDUAL DE NOTICIAS
# ============================================================

system_prompt_agente_web = """
Eres un Agente Analizador de Noticias especializado en verificación y comprensión informativa.

Siempre recibirás como entrada:
- URL de la noticia
- Título
- Fecha de publicación
- Texto completo de la noticia
- La consulta original del usuario (puede ser una afirmación o una pregunta)

Tu labor es:

1. Leer y analizar completamente la noticia proporcionada.
2. Si la consulta del usuario es una afirmación:
   - Determinar si la noticia RESPALDA, CONTRADICE o NO SE RELACIONA CLARAMENTE con dicha afirmación.
   - Explicar esta relación dentro del resumen.
3. Si la consulta del usuario es una pregunta:
   - Responderla usando exclusivamente la información contenida en la noticia.
4. Generar un resumen de la noticia en un máximo de 100 palabras.
5. Incluir siempre en la respuesta:
   - URL
   - Título
   - Fecha de publicación
   - Resumen de la noticia
   - Indicación explícita de si coincide o contradice la afirmación (si aplica)

IMPORTANTE:
- No inventes información.
- No utilices conocimiento externo: solo analiza lo que aparece en la noticia suministrada.
- Debes ser objetivo, conciso y claro.
- Si la noticia no contiene información suficiente para validar la afirmación o responder a la pregunta, indícalo explícitamente.

FORMATO DE RESPUESTA:

URL: <url>
Título: <título>
Fecha de publicación: <fecha>
Resumen (máx. 100 palabras): <resumen aquí>
Conclusión sobre la afirmación o respuesta a la pregunta:
<explicación final aquí>
"""

In [6]:
# ============================================================
# 5. CHAIN PARA ANALIZAR CADA NOTICIA
# ============================================================

analizador_noticia_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            system_prompt_agente_web,
        ),
        (
            "human",
            """
Consulta del usuario:
\"\"\"{consulta}\"\"\"

URL: {url}
Título: {title}
Fecha de publicación: {published_date}

Texto completo de la noticia:
\"\"\"{text}\"\"\"
""",
        ),
    ]
)


analizador_noticia_chain = (
    analizador_noticia_prompt
    | llm_2
    | StrOutputParser()
)

In [7]:
# ============================================================
# 6. LIMPIEZA DEL TEXTO
# ============================================================

def limpiar_texto(text: str) -> str:
    """
    Limpia el texto extraído de una noticia.

    - Elimina enlaces Markdown.
    - Elimina URLs sueltas.
    - Reemplaza saltos de línea.
    - Compacta espacios múltiples.
    """

    if not text:
        return ""

    # Eliminar enlaces Markdown: [texto](url) -> texto
    text = re.sub(
        r"\[([^\]]+)\]\([^)]+\)",
        r"\1",
        text
    )

    # Eliminar URLs sueltas
    text = re.sub(
        r"http\S+",
        "",
        text
    )

    # Reemplazar saltos de línea
    text = text.replace("\n", " ")

    # Compactar espacios
    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()

In [8]:
# ============================================================
# 7. TOOL DE BÚSQUEDA Y ANÁLISIS DE NOTICIAS
# ============================================================

@tool("buscar_noticias")
def web_scraping(consulta: str) -> str:
    """
    Busca y analiza noticias en medios peruanos como
    La República y El Comercio.

    Utiliza esta herramienta para:
    - buscar noticias;
    - investigar acontecimientos recientes;
    - responder preguntas de actualidad;
    - verificar afirmaciones;
    - realizar fact-checking.

    La entrada debe ser la pregunta o afirmación
    que se desea investigar.
    """

    try:

        # ====================================================
        # 1. BÚSQUEDA CON EXA
        # ====================================================

        result_web = exa.search(
            consulta,

            include_domains=[
                "larepublica.pe",
                "elcomercio.pe",
            ],

            num_results=3,

            type="auto",

            user_location="PE",

            contents={
                "text": {
                    "maxCharacters": 8000
                },

                # Permite utilizar contenido reciente.
                # Si la copia tiene más de 24 h,
                # Exa puede volver a recuperarla.
                "maxAgeHours": 24, #4380
            },
        )


        # ====================================================
        # 2. PROCESAR RESULTADOS
        # ====================================================

        data = []

        for resultado in result_web.results:

            texto = limpiar_texto(
                getattr(
                    resultado,
                    "text",
                    ""
                ) or ""
            )

            # Ignorar resultados sin contenido
            if not texto:
                continue


            data.append(
                {
                    "url": getattr(
                        resultado,
                        "url",
                        ""
                    ),

                    "title": getattr(
                        resultado,
                        "title",
                        ""
                    ),

                    "published_date": (
                        getattr(
                            resultado,
                            "published_date",
                            None
                        )
                        or "Fecha no disponible"
                    ),

                    "text": texto,
                }
            )


        # ====================================================
        # 3. VALIDAR RESULTADOS
        # ====================================================

        if not data:

            return (
                f"No se encontraron noticias con información suficiente "
                f"para la consulta: '{consulta}'. "
                "No es posible verificar la afirmación o responder "
                "la pregunta con las fuentes consultadas."
            )


        # ====================================================
        # 4. ANALIZAR CADA NOTICIA
        # ====================================================

        bloques = []


        for i, noticia in enumerate(
            data[:3],
            start=1
        ):

            try:

                analisis = analizador_noticia_chain.invoke(
                    {
                        "consulta": consulta,

                        "url": noticia["url"],

                        "title": noticia["title"],

                        "published_date": noticia[
                            "published_date"
                        ],

                        "text": noticia["text"],
                    }
                )


                bloques.append(
                    f"""
=== Noticia {i} ===
{analisis.strip()}
"""
                )


            except Exception as error:

                bloques.append(
                    f"""
=== Noticia {i} ===

No fue posible analizar esta noticia.

URL: {noticia["url"]}

Error técnico: {type(error).__name__}
"""
                )


        # ====================================================
        # 5. DEVOLVER RESULTADO AL AGENTE
        # ====================================================

        return "\n\n".join(
            bloques
        ).strip()


    except Exception as error:

        return (
            "Ocurrió un error durante la búsqueda de noticias. "
            f"Tipo de error: {type(error).__name__}. "
            "No fue posible obtener evidencia en este momento."
        )

In [10]:
# ============================================================
# 8. LISTA DE TOOLS
# ============================================================

tools = [
    web_scraping
]

In [12]:
# ============================================================
# 9. SYSTEM PROMPT PRINCIPAL DEL AGENTE
# ============================================================

fecha_actual = date.today().strftime(
    "%d/%m/%Y"
)


system_prompt_agente_noticias = f"""
Eres un Agente de Noticias especializado en análisis y verificación informativa. Tu función es responder únicamente
preguntas relacionadas con noticias, actualidad, empresas, sucesos relevantes y validar afirmaciones mediante herramientas
de búsqueda y verificación confiables.

Tu tarea es responder únicamente preguntas relacionadas con:
- noticias,
- actualidad,
- eventos recientes,
- empresas,
- sucesos relevantes,
- verificaciones de afirmaciones (fact-checking),
- análisis informativo,
- chistes con temática periodística.

No debes responder preguntas que no estén relacionadas con estos temas.


INSTRUCCIONES IMPORTANTES:
--------------------------

1. Debes ser lo más preciso, claro y útil posible.

2. Para responder preguntas relacionadas con hechos, noticias,
   actualidad o verificaciones:
   - NO debes basarte en tu conocimiento pre-entrenado para afirmar hechos.
   - Utiliza únicamente la información obtenida mediante las herramientas
     o la información incluida explícitamente en el contexto de la conversación.

3. Siempre responde en español.

4. Si no tienes suficiente información o la herramienta no devuelve evidencia,
   indícalo claramente.

5. Nunca inventes datos, cifras, declaraciones ni noticias.

6. Siempre que utilices una herramienta que devuelva noticias o análisis con URLs:
   - Basa tu respuesta en esa información.
   - Incluye al final una sección llamada "Fuentes consultadas"
     con las URLs más relevantes.

7. Si la información encontrada contradice la afirmación del usuario:
   - Debes explicarlo explícitamente.
   - Debes señalar qué fuentes contradicen la afirmación.

8. La fecha actual es {fecha_actual}.
   Usa esta referencia únicamente como contexto temporal.
   No inventes hechos futuros ni información que no aparezca
   en las fuentes consultadas.

9. Cuando la pregunta requiera información factual,
   reciente o una verificación, utiliza la herramienta
   de búsqueda disponible.

10. Si la pregunta está fuera del ámbito definido:
    - No utilices herramientas.
    - Indica brevemente que solo puedes responder consultas relacionadas
      con noticias, actualidad, empresas, sucesos relevantes,
      fact-checking, análisis informativo o temática periodística.

11. Para chistes con temática periodística:
    - Puedes responder directamente.
    - No utilices la herramienta de búsqueda.
    - No es necesario incluir fuentes.


HERRAMIENTAS DISPONIBLES:
-------------------------

Tienes acceso a una herramienta especializada para buscar y analizar
noticias de medios peruanos.

Cuando necesites verificar un hecho, investigar una noticia
o responder una pregunta de actualidad, utiliza la herramienta disponible.

El uso de las herramientas es interno.

No debes mostrar al usuario:
- razonamientos internos,
- llamadas de herramientas,
- nombres internos de herramientas,
- pasos técnicos del agente.


FORMATO DE LA RESPUESTA FINAL:
------------------------------

Redacta la respuesta principal en un único párrafo,
con el tono de un reportero o analista de noticias:
natural, fluido y directo.

La respuesta principal no debe superar las 100 palabras.

Integra de forma narrativa:
- la respuesta a la pregunta del usuario;
- el contexto necesario;
- la verificación de su afirmación, si corresponde;
- si la evidencia encontrada RESPALDA, CONTRADICE
  o NO PERMITE DETERMINAR claramente lo dicho por el usuario.

Cuando cites hechos, cifras o declaraciones,
menciona la fuente de manera natural y coloca la URL
entre paréntesis.

Ejemplo:

"Según un informe citado por El Comercio
(https://ejemplo.com/nota1), ..."

Si utilizaste la herramienta de búsqueda,
después del párrafo principal agrega:

Fuentes consultadas:
<URLs más relevantes utilizadas>

No incluyas URLs que no hayan sido obtenidas
mediante la herramienta.
"""

In [13]:
# ============================================================
# 10. MEMORIA CONVERSACIONAL
# ============================================================

memory = InMemorySaver()

In [14]:
# ============================================================
# 11. MIDDLEWARE
# ============================================================

middleware = [

    # Solo una búsqueda por pregunta del usuario
    ToolCallLimitMiddleware(
        tool_name="buscar_noticias",
        run_limit=1,
        exit_behavior="continue",
    ),

    # Evita ciclos excesivos entre modelo y tools
    ModelCallLimitMiddleware(
        run_limit=3,
        exit_behavior="end",
    ),
]

In [15]:
# ============================================================
# 12. CREACIÓN DEL AGENTE
# ============================================================

agent = create_agent(
    model=llm,

    tools=tools,

    system_prompt=system_prompt_agente_noticias,

    checkpointer=memory,

    middleware=middleware,
)

In [16]:
# ============================================================
# 14. FUNCIÓN PARA GENERAR RESPUESTAS
# ============================================================

def generate_response(
    user_input: str,
    session_id: str = "notebook-session"
) -> str:

    response = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": user_input,
                }
            ]
        },
        config={
            "configurable": {
                "thread_id": session_id
            }
        },
    )

    return response["messages"][-1].content

In [17]:
# ============================================================
# 15. CHAT INTERACTIVO
# ============================================================

SESSION_ID = "notebook-session"

while True:

    user_input = input("Pregunta: ").strip()

    if user_input.lower() in [
        "salir",
        "exit",
        "quit",
        "q",
    ]:
        print("\nAgente: Hasta luego 👋")
        break

    if not user_input:
        continue

    try:
        print("\n" + "=" * 80)
        print(f"👤 USUARIO:\n{user_input}")
        print("-" * 80)
        
        respuesta = generate_response(
            user_input=user_input,
            session_id=SESSION_ID,
        )

        
        print(f"🤖 AGENTE:\n{respuesta}")
        print("=" * 80 + "\n")

    except Exception as error:

        print("\n" + "=" * 80)
        print(f"👤 USUARIO:\n{user_input}")
        print("-" * 80)
        print(
            f"❌ ERROR:\n"
            f"{type(error).__name__}: {str(error)}"
        )
        print("=" * 80 + "\n")


👤 USUARIO:
ollanta humala es el actual presidente del peru?
--------------------------------------------------------------------------------
🤖 AGENTE:
Ollanta Humala no es el presidente actual del Perú; es un expresidente que gobernó entre 2011 y 2016. Recientemente, ha estado en el centro de procesos judiciales, pero no ocupa ningún cargo presidencial en la actualidad. La información más reciente confirma que Humala fue liberado tras la anulación de su condena, pero sigue siendo referido como exmandatario, no como presidente en funciones (El Comercio, https://elcomercio.pe/politica/justicia/ollanta-humala-en-libertad-por-orden-del-poder-judicial-y-tras-fallo-del-tribunal-constitucional-ultimas-noticia/).

Fuentes consultadas:
https://elcomercio.pe/politica/justicia/ollanta-humala-en-libertad-por-orden-del-poder-judicial-y-tras-fallo-del-tribunal-constitucional-ultimas-noticia/
https://elcomercio.pe/politica/justicia/ollanta-humala-defensa-legal-pide-al-poder-judicial-ejecutar-fallo-d